In [1]:
import pandas as pd
# import fpgrowth
from mlxtend.frequent_patterns import fpgrowth

In [2]:
df = pd.read_csv('Online_Retail_5000.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    5000 non-null   object 
 1   StockCode    5000 non-null   object 
 2   Description  4988 non-null   object 
 3   Quantity     5000 non-null   int64  
 4   InvoiceDate  5000 non-null   object 
 5   UnitPrice    5000 non-null   float64
 6   CustomerID   3795 non-null   float64
 7   Country      5000 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 312.6+ KB


In [ ]:
# đếm số lượng sản phẩm được mua trong mỗi hóa đơn
invoice_stockcode = df.groupby(['InvoiceNo', 'StockCode']).size().reset_index(name='Count')
invoice_stockcode.head()

,InvoiceNo,StockCode,Count
0,536365,21730,1
1,536365,22752,1
2,536365,71053,1
3,536365,84029E,1
4,536365,84029G,1


In [11]:
print(df['StockCode'].nunique())

1595


In [4]:
# tính tần suất của mỗi sản phẩm trong mỗi hóa đơn
stockcode_freq = df.groupby('StockCode')['InvoiceNo'].nunique().reset_index(name='Frequency')
stockcode_freq.head()

,StockCode,Frequency
0,10002,3
1,10124G,1
2,10125,1
3,10133,1
4,10135,2


In [5]:
total_invoices = df['InvoiceNo'].nunique()
stockcode_freq['Support'] = stockcode_freq['Frequency'] / total_invoices
stockcode_freq.head()

,StockCode,Frequency,Support
0,10002,3,0.010000
1,10124G,1,0.003333
2,10125,1,0.003333
3,10133,1,0.003333
4,10135,2,0.006667


In [6]:
stockcode_freq_sorted = stockcode_freq.sort_values(by='Support', ascending=False)
stockcode_freq_sorted.head(10)

,StockCode,Frequency,Support
949,22632,38,0.126667
1477,85123A,35,0.116667
950,22633,32,0.106667
1326,84029E,30,0.100000
1174,22961,27,0.090000
1033,22752,26,0.086667
1327,84029G,24,0.080000
1116,22866,23,0.076667
1217,37370,22,0.073333
1115,22865,22,0.073333


In [7]:
invoices = df['InvoiceNo'].unique()
print(f"Số lượng hóa đơn: {len(invoices)}")

Số lượng hóa đơn: 300


In [9]:
transactions = []
for invoice in invoices:
    items = df[df['InvoiceNo'] == invoice]['StockCode'].unique().tolist()
    transactions.append(items)
print(len(transactions))

300


In [10]:
print(transactions[:5])

[['85123A ', '71053 ', '84406B ', '84029G ', '84029E ', '22752 ', '21730 '], ['22633 ', '22632 '], ['84879 ', '22745 ', '22748 ', '22749 ', '22310 ', '84969 ', '22623 ', '22622 ', '21754 ', '21755 ', '21777 ', '48187 '], ['22960 ', '22913 ', '22912 ', '22914 '], ['21756 ']]


In [12]:
from mlxtend.preprocessing import TransactionEncoder
encoder = TransactionEncoder()
encoded_transactions = encoder.fit_transform(transactions)
encoded_df = pd.DataFrame(encoded_transactions, columns=encoder.columns_)
encoded_df.head()

,10002,10124G,10125,10133,10135,11001,15036,15044B,15056BL,15056N,...,90214M,90214R,90214S,90214V,BANK CHARGES,C2,D,DOT,M,POST
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [13]:
encoded_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Columns: 1595 entries, 10002  to POST 
dtypes: bool(1595)
memory usage: 467.4 KB


In [15]:
from mlxtend.frequent_patterns import association_rules
frequent_itemsets = fpgrowth(encoded_df, min_support=0.05, use_colnames=True)
fp_rules = association_rules(frequent_itemsets, metric="support", min_threshold=0.05)
fp_rules_sorted = fp_rules.sort_values(by='support', ascending=False)
print(f'Number of rules: {len(fp_rules_sorted)}')

Number of rules: 13604


In [16]:
top_rules = fp_rules_sorted.head(10)
for i, row in top_rules.iterrows():
    X = ', '.join(row['antecedents'])
    Y = ', '.join(row['consequents'])
    print(f"Rule: {X} -> {Y}")
    print(f"Support: {row['support']:.4f}")
    print(f"Confidence: {row['confidence']:.4f}")
    print(f"Lift: {row['lift']:.4f}")
    print("-" * 30)

Rule: 22633  -> 22632 
Support: 0.0867
Confidence: 0.8125
Lift: 6.4145
------------------------------
Rule: 22632  -> 22633 
Support: 0.0867
Confidence: 0.6842
Lift: 6.4145
------------------------------
Rule: 84029G  -> 84029E 
Support: 0.0733
Confidence: 0.9167
Lift: 9.1667
------------------------------
Rule: 84029E  -> 84029G 
Support: 0.0733
Confidence: 0.7333
Lift: 9.1667
------------------------------
Rule: 85123A  -> 84029E 
Support: 0.0700
Confidence: 0.6000
Lift: 6.0000
------------------------------
Rule: 84029G  -> 85123A 
Support: 0.0700
Confidence: 0.8750
Lift: 7.5000
------------------------------
Rule: 84029E  -> 85123A 
Support: 0.0700
Confidence: 0.7000
Lift: 6.0000
------------------------------
Rule: 85123A  -> 84029G 
Support: 0.0700
Confidence: 0.6000
Lift: 7.5000
------------------------------
Rule: 84029E  -> 85123A , 84029G 
Support: 0.0667
Confidence: 0.6667
Lift: 9.5238
------------------------------
Rule: 84029G  -> 85123A , 84029E 
Support: 0.0667
Confidenc